# 04 — Fine-tuned DistilBERT

Trains an actual transformer on the processed dataset. On an M1/M2/M3 Mac,
this uses the `mps` backend automatically (Apple Silicon GPU) — noticeably
faster than CPU. If it's still too slow, run this notebook in Google Colab
(free GPU) instead and copy the resulting `models/transformer_model` folder
back into this project.

In [ ]:
%pip install pandas torch transformers datasets scikit-learn matplotlib seaborn python-dotenv --quiet

In [ ]:
import sys, os
from pathlib import Path

# Make the project root importable so `from src... import ...` works
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

print("Project root:", PROJECT_ROOT)

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("MPS available:", torch.backends.mps.is_available())  # True on M1/M2/M3 Macs
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", DEVICE)

In [ ]:
import pandas as pd
from src.config import PROCESSED_DATA_DIR, LABEL2ID, ID2LABEL

df = pd.read_csv(PROCESSED_DATA_DIR / "dataset.csv")
df["label"] = df["sentiment"].map(LABEL2ID)
print(f"Loaded {len(df)} rows")
df.head()

In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["full_text"], truncation=True, padding="max_length", max_length=128)

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True)).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True)).map(tokenize, batched=True)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from src.config import MODELS_DIR

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "f1_macro": f1_score(labels, preds, average="macro")}

training_args = TrainingArguments(
    output_dir=str(MODELS_DIR / "checkpoints"),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,   # drop to 8 if you hit an out-of-memory error on 8GB M1
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
from src.evaluate import print_report, plot_confusion_matrix

predictions = trainer.predict(test_ds)
y_pred_ids = np.argmax(predictions.predictions, axis=-1)
y_pred_transformer = [ID2LABEL[i] for i in y_pred_ids]
y_true_transformer = test_df["sentiment"].tolist()

print_report(y_true_transformer, y_pred_transformer, model_name="Fine-tuned DistilBERT")
plot_confusion_matrix(y_true_transformer, y_pred_transformer, model_name="DistilBERT", cmap="Purples")

transformer_accuracy = accuracy_score(y_true_transformer, y_pred_transformer)
transformer_f1 = f1_score(y_true_transformer, y_pred_transformer, average="macro")
print(f"\nAccuracy: {transformer_accuracy:.3f} | Macro F1: {transformer_f1:.3f}")

In [ ]:
import json

model.save_pretrained(MODELS_DIR / "transformer_model")
tokenizer.save_pretrained(MODELS_DIR / "transformer_model")

with open(MODELS_DIR / "transformer_metrics.json", "w") as f:
    json.dump({"accuracy": transformer_accuracy, "f1_macro": transformer_f1}, f)

print("Saved model, tokenizer, and metrics.")

Next: open **05_model_comparison.ipynb** to compare both models.